# Algoritmo de colonia de hormigas para rutas óptimas entre nodos

### Adaptado a partir de

<font size="4"> Fernando Jiménez Barrionuevo, Gracia Sánchez Carpena (UMU) — "Algoritmo de colonia de hormigas para selección de atributos"

### Descripción

<font size="4"> Algoritmo de colonia de hormigas (ACO) aplicado a la búsqueda de la ruta más corta que recorre un conjunto de nodos seleccionados (puntos rojos) sobre la red vial real. Se parte de una matriz de distancias ya calculada con caminos más cortos (Dijkstra) sobre el grafo de OSMnx/NetworkX.

<font size="4"> A diferencia del problema de selección de atributos (donde cada hormiga elige un **subconjunto no ordenado** de elementos), aquí cada hormiga construye una **secuencia ordenada** (permutación) de todos los nodos, por lo que:
- la feromona es una **matriz 2D** `tao[i][j]` (deseabilidad de ir del nodo i al nodo j),
- la heurística `eta[i][j] = 1/distancia[i][j]` es **estática**, calculada una sola vez a partir de la matriz de distancias,
- la probabilidad de elegir el siguiente nodo depende del **nodo actual** de la hormiga.

### Componentes

- <font size="3"> Datos: matriz de distancias (metros) entre los nodos seleccionados, calculada previamente con caminos más cortos sobre la red vial real
- <font size="3"> Problema: TSP (ruta cerrada que visita todos los nodos una vez y vuelve al origen)
- <font size="3"> Coste a minimizar: distancia total de la ruta

---

### Requerimientos

In [1]:
import numpy as np
import pandas as pd
from tqdm import tqdm
import matplotlib.pyplot as plt

### Lectura de la matriz de distancias
#### Input:
- $file\_path$: ruta al csv con la matriz de distancias (la generada con Dijkstra sobre el grafo vial)

#### Output:
- $dist\_matrix$: matriz $n\times n$ de distancias (metros) entre los nodos seleccionados
- $etiquetas$: identificadores de los nodos (columna `id` de tu csv original), en el mismo orden que la matriz

In [8]:
def LoadDistanceMatrix(file_path):
    df = pd.read_csv(file_path, index_col=0)
    etiquetas = df.index.tolist()
    dist_matrix = df.values.astype(float)
    return dist_matrix, etiquetas

file_path = "matriz_distancias.csv"  # la matriz que ya generaste con Dijkstra
dist_matrix, etiquetas = LoadDistanceMatrix(file_path)
n = dist_matrix.shape[0]
print("Número de nodos:", n)

FileNotFoundError: [Errno 2] No such file or directory: 'matriz_distancias.csv'

### Parámetros del algoritmo de colonia de hormigas

- $N$: Número de hormigas en la colonia
- $T$: Número de iteraciones
- $rho$: Tasa de evaporación. Igual que en el notebook original, valores por debajo de 0.5 o por encima de 0.8 ralentizan la convergencia; óptimo habitual alrededor de 0.7.
- $alpha$: Importancia del rastro de feromona. Rango habitual $1\leq alpha\leq1.5$.
- $beta$: Importancia de la heurística (visibilidad, $1/distancia$). Valores altos ($beta>5$) pueden estancar el algoritmo en soluciones subóptimas.
- $Q$: Factor multiplicativo del depósito de feromona.
- $cerrado$: si `True`, la ruta vuelve al nodo de partida (TSP cerrado); si `False`, termina en el último nodo visitado.
- $random\_state$: Semilla para reproducibilidad

In [ ]:
N = 30
T = 200
alpha = 1.0
beta = 3.0
rho = 0.5
Q = 100
cerrado = True
random_state = 1

### Definición del problema de optimización

$\textit{Minimizar} \ f(\textbf{x}) = \sum_{l=0}^{n-2} d(x_l,x_{l+1}) \ (+ \ d(x_{n-1},x_0) \ \text{si es cerrado})$

- $\textbf{x}=(x_0,\ldots,x_{n-1})$ es una **permutación** de los $n$ nodos seleccionados (orden de visita)
- $d(i,j)$: distancia del camino más corto por carretera entre el nodo $i$ y el nodo $j$ (de la matriz de distancias)

In [ ]:
def f(x, dist_matrix, cerrado):
    # x: array de índices (permutación de 0..n-1)
    coste = sum(dist_matrix[x[l], x[l+1]] for l in range(len(x)-1))
    if cerrado:
        coste += dist_matrix[x[-1], x[0]]
    return coste

### Definición de hormiga
- $n$: Número de nodos a visitar (todos, a diferencia del notebook original donde $n$ variaba entre $n\_L$ y $n\_U$)

In [ ]:
class Ant:
    def __init__(self, n):
        self.x = np.full(n, -1)
        self.f = np.inf
    def __lt__(self, ant):
        return self.f < ant.f

### Inicialización de las feromonas y de la heurística
- $tao$: matriz $n\times n$ de feromona (deseabilidad de ir de $i$ a $j$), inicializada a un valor pequeño constante, igual que en el notebook original
- $eta$: heurística estática $1/d(i,j)$, calculada una sola vez (a diferencia del notebook original, donde se recalculaba el ACC en cada paso de construcción — aquí es mucho más barato porque no hay que reentrenar ningún modelo)

In [ ]:
def InitializePheromones(n):
    return np.full((n, n), 1E-3)

def InitializeHeuristic(dist_matrix):
    n = dist_matrix.shape[0]
    eta = 1 / (dist_matrix + np.eye(n) * 1e-10)
    np.fill_diagonal(eta, 0.0)
    return eta

### Construcción de la colonia de hormigas

Cada hormiga construye su ruta paso a paso: parte de un nodo aleatorio y, en cada paso, elige el siguiente nodo entre los no visitados con probabilidad proporcional a $tao[actual][j]^{alpha} \cdot eta[actual][j]^{beta}$ (igual filosofía que `ConstructAnts` del notebook original, pero aquí la probabilidad depende del nodo actual, no es independiente paso a paso).

In [ ]:
def ConstructAnts(P, alpha, beta, tao, eta):
    n = tao.shape[0]
    for I in P:
        inicio = np.random.randint(n)
        I.x[0] = inicio
        no_visitados = list(range(n))
        no_visitados.remove(inicio)
        for l in range(1, n):
            actual = I.x[l-1]
            pesos = np.array([tao[actual][j]**alpha * eta[actual][j]**beta for j in no_visitados])
            suma = pesos.sum()
            if suma == 0:
                siguiente = np.random.choice(no_visitados)
            else:
                p = pesos / suma
                siguiente = np.random.choice(no_visitados, p=p)
            I.x[l] = siguiente
            no_visitados.remove(siguiente)

### Función de evaluación

In [ ]:
def EvaluateAnts(P, dist_matrix, cerrado):
    for I in P:
        I.f = f(I.x, dist_matrix, cerrado)

### Copia de hormiga

In [ ]:
def Copia(ant1, ant2):
    ant1.x[:] = ant2.x[:]
    ant1.f = ant2.f

### Actualización de las feromonas

Igual que en el notebook original (evaporación + depósito proporcional a $Q/f$), pero aquí el depósito se hace sobre cada **arista** (par ordenado) de la ruta recorrida, en vez de sobre cada atributo seleccionado.

In [ ]:
def UpdatePheromones(P, tao, rho, Q, cerrado):
    tao *= (1 - rho)  # evaporación
    for I in P:
        x = I.x
        deposito = Q / I.f
        for l in range(len(x)-1):
            a, b = x[l], x[l+1]
            tao[a][b] += deposito
            tao[b][a] += deposito
        if cerrado:
            a, b = x[-1], x[0]
            tao[a][b] += deposito
            tao[b][a] += deposito

### Progreso del algoritmo

In [ ]:
def IniciaProgreso(P, best_global, T):
    best_outputs = [0]*(T+1)
    best_outputs_global = [0]*(T+1)
    progression_bar = tqdm()
    ActualizaProgreso(P, best_global, best_global, 0, best_outputs, best_outputs_global, progression_bar)
    return best_outputs, best_outputs_global, progression_bar

def ActualizaProgreso(P, best_global, best_individual, iter, best_outputs, best_outputs_global, progression_bar):
    progression_bar.set_description("Iteration: %s Best global (m): %.1f Best iter (m): %.1f" % (str(iter), best_global.f, best_individual.f))
    best_outputs[iter] = best_individual.f
    best_outputs_global[iter] = best_global.f
    progression_bar.update(1)

### Algoritmo de colonia de hormigas

In [ ]:
def AntsColonyOptimization(N, T, alpha, beta, rho, Q, dist_matrix, cerrado, random_state):
    n = dist_matrix.shape[0]

    assert N>=1, "El número de hormigas debe ser mayor o igual que uno."
    assert T>1, "El número de iteraciones debe ser mayor que uno."
    assert rho>0, "La tasa de evaporación debe ser mayor que cero."
    assert rho<1, "La tasa de evaporación debe ser menor que uno."

    np.random.seed(random_state)

    tao = InitializePheromones(n)
    eta = InitializeHeuristic(dist_matrix)

    P = [Ant(n) for _ in range(N)]
    ConstructAnts(P, alpha, beta, tao, eta)
    EvaluateAnts(P, dist_matrix, cerrado)
    best_ant = Ant(n)
    Copia(best_ant, min(P))

    best_outputs, best_outputs_global, progression_bar = IniciaProgreso(P, best_ant, T)
    for iter in range(1, T+1):
        UpdatePheromones(P, tao, rho, Q, cerrado)
        ConstructAnts(P, alpha, beta, tao, eta)
        EvaluateAnts(P, dist_matrix, cerrado)
        best_ant_iter = min(P)
        if best_ant_iter < best_ant:
            Copia(best_ant, best_ant_iter)
        ActualizaProgreso(P, best_ant, best_ant_iter, iter, best_outputs, best_outputs_global, progression_bar)
    return best_outputs, best_outputs_global, best_ant.f, best_ant.x

### Ejecución del algoritmo de colonia de hormigas

In [ ]:
best_outputs, best_outputs_global, best_fitness, best_route = AntsColonyOptimization(
    N, T, alpha, beta, rho, Q, dist_matrix, cerrado, random_state
)

### Visualización de la evolución del algoritmo

In [ ]:
def VisualizaEvolucion(best_outputs, ylabel):
    fig, ax = plt.subplots()
    ax.set_title("Algorithm evolution")
    plt.plot(best_outputs)
    plt.xlabel("Iteration")
    plt.ylabel(ylabel)
    plt.show()

VisualizaEvolucion(best_outputs, "Distancia total (m) \n Mejor solución de la colonia actual")
VisualizaEvolucion(best_outputs_global, "Distancia total (m) \n Mejor solución global")

### Solución del algoritmo

In [ ]:
ruta_con_ids = [etiquetas[i] for i in best_route]
print("Solución del algoritmo (orden de visita)")
print("Ruta (ids de nodo):", ruta_con_ids)
print("Distancia total de la ruta (m):", round(best_fitness, 1))
print("Ruta cerrada (vuelve al origen):", cerrado)